# Project country BMR onto the 0.25° grid

Joins `{GBD_version}_BMR_Country_{health_VAR}_newlabels_1990-2009.nc` (one value per country name) to `country_id_mask_0_25.nc` (one integer ID per cell) **via the name→ID table in `country_location_ids.json`**. Join key is the integer ID, with explicit checks that nothing is silently dropped or mismatched.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [12]:
# Load country id mask
mask_file = "country_id_mask_0.25.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataset(mask_path)

# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

In [4]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [5]:
GBD_version = "GBD23"

In [17]:
# === Main loop ===
for health_VAR in health_vars:
    # Load mean BMR by country
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path).sel(quantile="mean")  # central estimates

    # Create dataframe with country, bmr, cid
    df = pd.DataFrame({
        "country": [str(c) for c in bmr["country"].values],
        "bmr":     bmr.values,
    })
    df["country_id"] = df["country"].map(country_to_id)
    df["country_id"] = df["country_id"].astype(int)

    # Checks:
    unmatched = df[df["country_id"].isna()]
    assert unmatched.empty, f"BMR countries with no ID in the JSON: {unmatched['country'].tolist()}"
    assert df["country"].is_unique, f"Duplicate country names in {bmr_file}"
    assert df["country_id"].is_unique, "Two BMR countries map to the same ID"
    assert df["bmr"].notna().all(), "NaN BMR values present"
    
    unused_json = sorted(set(country_to_id) - set(df["country"]))
    print(f"{len(df)} BMR countries, all matched to an ID. JSON entries without BMR: {unused_json}")

    ids = masks["country_id"].values  # (lat, lon), float64 with NaN
    valid = np.isfinite(ids)
    ids_int = np.where(valid, ids, -1).astype(np.int64)  # -1 = no country
    # All IDs must be integers
    assert np.all(ids[valid] == np.round(ids[valid])), "Non-integer IDs in mask"
    
    # Every ID in the mask must exist in the JSON, otherwise a cell would get no value for the wrong reason
    mask_ids = set(np.unique(ids_int[valid]).tolist())
    unknown = mask_ids - set(country_to_id.values())
    assert not unknown, f"Mask contains IDs not in the JSON: {sorted(unknown)}"
    
    # Vectorised lookup via a dense array indexed by ID
    lut = np.full(max(country_to_id.values()) + 1, np.nan)
    lut[df["country_id"].values] = df["bmr"].values
    
    bmr_grid = np.full(ids.shape, np.nan)
    bmr_grid[valid] = lut[ids_int[valid]]
    
    bmr_da = xr.DataArray(
        bmr_grid, coords=masks["country_id"].coords, dims=masks["country_id"].dims,
        name="BMR",
        attrs={**bmr.attrs,
               "grid_note": "Value of the country that covers the largest share of each 0.25 deg cell; NaN where no country.",
               "id_source": "country_location_ids.json"},
    )

    # Save with country ids and country fractions for later
    out = xr.Dataset({
        "BMR": bmr_da,
        "country_id": masks["country_id"],
        "country_id_frac": masks["country_id_frac"],
    })

    out_file = f"{GBD_version}_BMR_Global_Country_Map_era5_{health_VAR}_1990-2009.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    print(f"Saving to {out_path}")
    out.to_netcdf(out_path)

print("All processing complete.")

204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_CHRONIC_KIDNEY_DISEASE_1990-2009.nc
204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_CARDIOMYOPATHY_MYOCARDITIS_1990-2009.nc
204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_HYPERTENSIVE_HEART_DISEASE_1990-2009.nc
204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_ISCHEMIC_HEART_DISEASE_1990-2009.nc
204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_STROKE_1990-2009.nc
204 BMR countries, all matched to an ID. JSON entries without BMR: []
Saving to /glade/work/awells